<a href="https://colab.research.google.com/github/rund-omari/Data.Eng3/blob/main/lectures/Lecture_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import html
df = pd.read_csv("/content/SDK_NLP_Customer_Reviews_Practice.csv")
df

,review_id,review_date,product_category,channel,rating,review_text
0,REV-0001,2026-01-04,Beauty,Marketplace,1,Never ordering again. Support was unhelpful.
1,REV-0002,2026-01-06,Accessories,Mobile App,2,The color is different from the photo.
2,REV-0003,2026-01-08,Electronics,Website,2,I need to contact support@example.com today.
3,REV-0004,2026-01-10,Home,Marketplace,5,This product is better than expected.
4,REV-0005,2026-01-12,Fashion,Mobile App,4,Order arrived in Amman within 2 days.
...,...,...,...,...,...,...
115,REV-0116,2026-08-22,Beauty,Mobile App,3,This is okay... nothing special.
116,REV-0117,2026-08-24,Accessories,Website,3,Product quality is good; shipping was SLOW.
117,REV-0118,2026-08-26,Electronics,Marketplace,5,Very good value for 25 JOD.
118,REV-0119,2026-08-28,Home,Mobile App,1,"No, I would not recommend it."


In [ ]:
df["review_text_raw"] = df["review_text"]
df[["review_text_raw"]].head()

,review_text_raw
0,Never ordering again. Support was unhelpful.
1,The color is different from the photo.
2,I need to contact support@example.com today.
3,This product is better than expected.
4,Order arrived in Amman within 2 days.


In [ ]:
df['review_text_raw'].sample(10)

,review_text_raw
3,This product is better than expected.
50,The delivery was VERY late!!! 😡 But the produc...
89,Easy return process and friendly service.
107,Irbid delivery took longer than expected.
113,AMAZING quality!! Would buy again 😊
43,This product is better than expected.
31,Great product!!! But the box was damaged.
2,I need to contact support@example.com today.
83,This product is better than expected.
14,Worst delivery experience: 12 days late.


In [ ]:
" ".strip()

''

In [ ]:
text = df["review_text"].astype("string")
blank = text.isna() | text.str.strip().eq("").fillna(False)
print("Missing or blank:", blank.sum())

Missing or blank: 8


In [ ]:
df[df["review_text"].str.strip().eq("").fillna(False)]

,review_id,review_date,product_category,channel,rating,review_text,review_text_raw
36,REV-0037,2026-03-17,Accessories,Marketplace,3,,
82,REV-0083,2026-06-17,Electronics,Mobile App,2,,


In [ ]:
df["char_count"] = text.str.len()
df["word_count"] = text.str.split().str.len().fillna(0)
df[["char_count", "word_count"]].describe()

/tmp/ipykernel_789/2133239261.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["word_count"] = text.str.split().str.len().fillna(0)


,char_count,word_count
count,114.0,120.000000
mean,36.991228,5.883333
std,8.791081,2.038220
min,2.0,0.000000
25%,33.25,5.000000
50%,39.0,6.000000
75%,41.0,7.000000
max,56.0,11.000000


In [ ]:
txt = "my order arrived after 34 days!!!"
numbers = re.findall(r"\d+" , txt)
numbers

['34']

In [ ]:
import re
from nltk.tokenize import RegexpTokenizer

text = "The delivery wasn't fast!!!"

print(text.split())
print(re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower()))

tokenizer = RegexpTokenizer(r"[a-z]+(?:'[a-z]+)?")
print(tokenizer.tokenize(text.lower()))
def clean_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)   # basic HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.lower()
    return " ".join(text.split())
def tokenize_text(text):
    tokenizer = RegexpTokenizer(r"[a-z]+(?:'[a-z]+)?")
    return(tokenizer.tokenize(text))
stop_words = {
    "the", "is", "was", "a", "an", "and", "to",
    "for", "of", "in", "on", "it", "this", "that",
    "very", "at", "with", "my", "i", "are"}
def remove_stopwords(text):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    print(words)
    return [w for w in words if w not in stop_words]




['The', 'delivery', "wasn't", 'fast!!!']
['the', 'delivery', "wasn't", 'fast']
['the', 'delivery', "wasn't", 'fast']


In [ ]:
tokenizer = RegexpTokenizer(r"[a-z]+(?:'[a-z]+)?")

def clean_text(value):
    if pd.isna(value):
        return ""

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.lower()

    return " ".join(text.split())


def tokenize_text(text):
    return tokenizer.tokenize(text)


stop_words = {
    "the", "is", "was", "a", "an", "and", "to",
    "for", "of", "in", "on", "it", "this", "that",
    "very", "at", "with", "my", "i", "are"
}

def remove_stopwords(words):
    return [w for w in words if w not in stop_words]


def preprocess_text(value):
    cleaned = clean_text(value)
    tokens = tokenize_text(cleaned)
    filtered = remove_stopwords(tokens)

    return filtered


df["review_text_raw"] = df["review_text"].apply(preprocess_text)

df[["review_text", "review_text_raw"]].head()


,review_text,review_text_raw
0,Never ordering again. Support was unhelpful.,"[never, ordering, again, support, unhelpful]"
1,The color is different from the photo.,"[color, different, from, photo]"
2,I need to contact support@example.com today.,"[need, contact, support, example, com, today]"
3,This product is better than expected.,"[product, better, than, expected]"
4,Order arrived in Amman within 2 days.,"[order, arrived, amman, within, days]"
